# Neuro-Semantic Suicide Risk Detection - Demo

This notebook demonstrates how to load a trained model, run inference on a sample EEG and text, and generate explainability maps using LRP and DeepLIFT.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from captum.attr import DeepLIFT, LayerIntegratedGradients
import sys
sys.path.append('..')
from models.eeg_encoder import EEGEncoder
from models.text_encoder import TextEncoder
from models.fusion import HypernetworkFusion
from models.classifier import ClassifierHead
from data.preprocessing import compute_spectrogram, compute_adjacency

# Configuration (same as in config.yaml)
config = {
    'd_model': 512,
    'nhead': 8,
    'num_layers': 4,
    'eeg_channels': 64,
    'bert_model': 'emilyalsentzer/Bio_ClinicalBERT',
    'temperature': 0.07
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Load pre-trained model
eeg_encoder = EEGEncoder(config).to(device)
text_encoder = TextEncoder(config).to(device)
hyperfusion = HypernetworkFusion(config['d_model']).to(device)
classifier = ClassifierHead(config['d_model']).to(device)

# Load checkpoint (adjust path)
# checkpoint = torch.load('checkpoints/best.pth', map_location=device)
# eeg_encoder.load_state_dict(checkpoint['eeg_encoder'])
# text_encoder.load_state_dict(checkpoint['text_encoder'])
# hyperfusion.load_state_dict(checkpoint['hyperfusion'])
# classifier.load_state_dict(checkpoint['classifier'])

eeg_encoder.eval()
text_encoder.eval()
hyperfusion.eval()
classifier.eval()

In [ ]:
# Sample input (random data for demonstration)
sample_eeg = np.random.randn(64, 500)  # 64 channels, 500 time points
sample_spectrogram = compute_spectrogram(sample_eeg, fs=250)
sample_adj = compute_adjacency(sample_eeg)

sample_text = "I feel hopeless and worthless"

# Convert to tensors
eeg_signal = torch.tensor(sample_eeg).unsqueeze(0).float().to(device)  # (1,64,500)
eeg_spec = torch.tensor(sample_spectrogram).unsqueeze(0).float().to(device)  # (1,64,freq,time)
eeg_adj = torch.tensor(sample_adj).unsqueeze(0).float().to(device)  # (1,64,64)

with torch.no_grad():
    z_eeg = eeg_encoder(eeg_signal, eeg_spec, eeg_adj)
    z_text = text_encoder([sample_text])
    # For uncertainty and complexity (simplified)
    u_eeg = torch.ones(1, device=device) * 0.1
    sigma_eeg = torch.ones(1, device=device) * 0.1
    u_text = torch.ones(1, device=device) * 0.1
    sigma_text = torch.ones(1, device=device) * 0.1
    z_shared = hyperfusion(z_eeg, z_text, u_eeg, sigma_eeg, u_text, sigma_text)
    bin_logits, ord_logits = classifier(z_shared)
    prob_bin = torch.softmax(bin_logits, dim=1)
    prob_ord = torch.softmax(ord_logits, dim=1)

print(f"Binary risk probability: {prob_bin.cpu().numpy()}")
print(f"Ordinal risk probability (0-3): {prob_ord.cpu().numpy()}")

In [ ]:
# Explainability: DeepLIFT for EEG
def forward_eeg(x):
    # x is eeg_signal (1,64,500)
    spec = torch.tensor(compute_spectrogram(x[0].cpu().numpy())).unsqueeze(0).float().to(device)
    adj = torch.tensor(compute_adjacency(x[0].cpu().numpy())).unsqueeze(0).float().to(device)
    z = eeg_encoder(x, spec, adj)
    z_shared = hyperfusion(z, z_text, u_eeg, sigma_eeg, u_text, sigma_text)
    out = classifier(z_shared)[0]  # bin_logits
    return out

# Use a baseline (e.g., zeros)
baseline_eeg = torch.zeros_like(eeg_signal)
deeplift_eeg = DeepLIFT(forward_eeg)
attributions = deeplift_eeg.attribute(eeg_signal, baselines=baseline_eeg, target=1)  # target=1 for high risk
attributions = attributions.squeeze().cpu().detach().numpy()  # (64,500)

# Plot top 10 channels importance (sum over time)
channel_importance = attributions.sum(axis=1)
top_channels = np.argsort(channel_importance)[-10:]
plt.figure(figsize=(10,4))
plt.bar(range(10), channel_importance[top_channels])
plt.xlabel('Channel Index')
plt.ylabel('DeepLIFT Attribution')
plt.title('Top 10 EEG Channels by Importance')
plt.show()

In [ ]:
# Explainability for text using Integrated Gradients
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config['bert_model'])

def forward_text(texts):
    # texts is a list of strings
    z = text_encoder(texts)
    z_shared = hyperfusion(z_eeg, z, u_eeg, sigma_eeg, u_text, sigma_text)
    out = classifier(z_shared)[0]
    return out

ig = LayerIntegratedGradients(forward_text, text_encoder.bert.embeddings)
tokens = tokenizer(sample_text, return_tensors='pt', max_length=128, truncation=True)
attributions_text = ig.attribute(tokens['input_ids'].to(device),
                                 baselines=torch.zeros_like(tokens['input_ids']).to(device),
                                 target=1,
                                 n_steps=50)
attributions_text = attributions_text.squeeze().cpu().detach().numpy()

# Map to tokens
token_strs = tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])
for token, attr in zip(token_strs[:10], attributions_text[:10]):
    print(f"{token}: {attr:.4f}")